In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

# =====================================================================
# 1. LOAD DATASET & SPLITTING RE-RUN (Agar Variabel Terbaca di Memori)
# =====================================================================
file_dataset = "../data/processed/cases.csv"
folder_results = "../data/results"
os.makedirs(folder_results, exist_ok=True)

if os.path.exists(file_dataset):
    df = pd.read_csv(file_dataset)
    # Lakukan splitting ulang dengan random_state yang SAMA (42) agar pembagian datanya identik
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    print(f"✅ Sukses memuat dataset!")
    print(f" - Jumlah Data Train (Kasus Lama) : {len(df_train)}")
    print(f" - Jumlah Data Test (Kasus Baru)  : {len(df_test)}")
else:
    raise FileNotFoundError("❌ File cases.csv tidak ditemukan!")

# =====================================================================
# 2. SEED/RE-TRAIN MODEL RETRIEVAL DARI TAHAP 3
# =====================================================================
print("🔄 Mempersiapkan model retrieval TF-IDF + SVM...")
tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(df_train['text_full'])

model_svm = SVC(kernel='linear', probability=True, random_state=42)
model_svm.fit(X_train_tfidf, df_train['case_id'])
print("✅ Model retrieval siap digunakan!")

# Fungsi retrieve disalin ke sini agar fungsi predict_outcome di Cell 2 bisa memanggilnya
def retrieve(query: str, pendekatan: str = "tfidf_svm", k: int = 5):
    query_clean = query.lower()
    if pendekatan == "tfidf_svm":
        query_vec = tfidf_vectorizer.transform([query_clean])
        probabilitas = model_svm.predict_proba(query_vec).flatten()
        top_k_indices = probabilitas.argsort()[::-1][:k]
        
        hasil_retrieval = []
        for idx in top_k_indices:
            cid = model_svm.classes_[idx]
            row = df_train[df_train['case_id'] == cid].iloc[0]
            hasil_retrieval.append({
                'case_id': row['case_id'],
                'no_perkara': row['no_perkara'],
                'score': round(float(probabilitas[idx]), 4),
                'pasal': row['pasal'],
                'pihak': row['pihak']
            })
        return hasil_retrieval
    else:
        raise ValueError("Pendekatan di luar tfid_svm memerlukan inisialisasi model terkait.")

# =====================================================================
# 3. i. EKSTRAK SOLUSI DARI KASUS LAMA (Inti Langkah Kerja Tahap 4)
# =====================================================================
# Memetakan case_id ke kolom argumen_hukum_utama
case_solutions = dict(zip(df_train['case_id'], df_train['argumen_hukum_utama']))

print("-" * 60)
print(f"✅ TAHAP 4 CELL 1 BERHASIL DI-LOAD!")
print(f"Total solusi yang siap dipakai ulang: {len(case_solutions)} kasus.")
print("-" * 60)

✅ Sukses memuat dataset!
 - Jumlah Data Train (Kasus Lama) : 38
 - Jumlah Data Test (Kasus Baru)  : 10
🔄 Mempersiapkan model retrieval TF-IDF + SVM...
✅ Model retrieval siap digunakan!
------------------------------------------------------------
✅ TAHAP 4 CELL 1 BERHASIL DI-LOAD!
Total solusi yang siap dipakai ulang: 38 kasus.
------------------------------------------------------------


c:\Users\Nabila Aurellya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:785: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  check_classification_targets(y)
c:\Users\Nabila Aurellya\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


In [2]:
def predict_outcome(query: str, pendekatan_retrieval: str = "tfidf_svm"):
    """
    Fungsi untuk memprediksi solusi kasus baru (Langkah iii)
    Menggunakan pendekatan Weighted Similarity dari hasil top-k retrieval.
    """
    # 1. Panggil fungsi retrieve dari Tahap 3 (mengambil top-5 kasus terdekat)
    top_k = retrieve(query, pendekatan=pendekatan_retrieval, k=5)
    
    if not top_k:
        return "TIDAK DITEMUKAN SOLUSI RELEVAN", [], []
    
    # 2. Ambil solusi teks dan skor kemiripan dari top-k kasus tersebut
    top_5_case_ids = [item['case_id'] for item in top_k]
    top_5_scores = [item['score'] for item in top_k]
    
    # ii. Algoritma Prediksi (Weighted Similarity):
    # Solusi yang dipilih adalah milik kasus dengan skor similarity tertinggi (indeks 0 hasil retrieve)
    indeks_terbaik = np.argmax(top_5_scores)
    case_id_terpilih = top_5_case_ids[indeks_terbaik]
    
    # Ambil teks amar putusan penuh dari kamus solusi kita
    predicted_solution = case_solutions[case_id_terpilih]
    
    return predicted_solution, top_5_case_ids, top_5_scores

print("✅ Fungsi predict_outcome() dengan Algoritma Weighted Similarity berhasil dimuat!")

✅ Fungsi predict_outcome() dengan Algoritma Weighted Similarity berhasil dimuat!


In [3]:
# iv. Demo Manual menggunakan 5 Kasus Baru dari data_test
hasil_prediksi_list = []

print("🚀 Memulai Demo Manual Prediksi Putusan Kasus Baru...\n")

# Ambil 5 data uji dari df_test
df_demo = df_test.head(5)

for idx, row in enumerate(df_demo.itertuples()):
    query_id = f"Q_NEW_{idx+1:03d}"
    query_teks = row.ringkasan_fakta
    
    # Jalankan fungsi prediksi
    prediksi_solusi, top_5_ids, top_5_scores = predict_outcome(query_teks, pendekatan_retrieval="tfidf_svm")
    
    # Tampilkan log demo manual di layar jupyter
    print(f"🔹 Query ID         : {query_id}")
    print(f"🔹 Kasus Asli       : {row.no_perkara}")
    print(f"🔹 Top 5 Kasus Mirip: {', '.join(top_5_ids)}")
    print(f"🔹 Prediksi Amar    : {prediksi_solusi[:120]}...")
    print("-" * 60)
    
    # Simpan data ke list untuk diubah jadi CSV sesuai format silabus
    hasil_prediksi_list.append({
        "query_id": query_id,
        "predicted_solution": prediksi_solusi,
        "top_5_case_ids": ", ".join(top_5_ids) # Menggabungkan top 5 ID dipisah koma
    })

# v. Output: Simpan ke format terstruktur CSV
df_predictions = pd.DataFrame(hasil_prediksi_list)
path_predictions_csv = os.path.join(folder_results, "predictions.csv")
df_predictions.to_csv(path_predictions_csv, index=False, encoding='utf-8')

print(f"File laporan prediksi berhasil diekspor ke: {path_predictions_csv}")

🚀 Memulai Demo Manual Prediksi Putusan Kasus Baru...

🔹 Query ID         : Q_NEW_001
🔹 Kasus Asli       : 3187 K/PDT/2024
🔹 Top 5 Kasus Mirip: case_003, case_037, case_012, case_050, case_023
🔹 Prediksi Amar    : 1. menolak permohona n kasasi dari para pemohon kasasi 1. reki manuho dan 2. riselyu manuho tersebut 2. menghukum para p...
------------------------------------------------------------
🔹 Query ID         : Q_NEW_002
🔹 Kasus Asli       : 5212 K/PDT/2024
🔹 Top 5 Kasus Mirip: case_003, case_050, case_012, case_041, case_032
🔹 Prediksi Amar    : 1. menolak permohona n kasasi dari para pemohon kasasi 1. reki manuho dan 2. riselyu manuho tersebut 2. menghukum para p...
------------------------------------------------------------
🔹 Query ID         : Q_NEW_003
🔹 Kasus Asli       : 3156 K/PDT/2025
🔹 Top 5 Kasus Mirip: case_003, case_037, case_023, case_017, case_012
🔹 Prediksi Amar    : 1. menolak permohona n kasasi dari para pemohon kasasi 1. reki manuho dan 2. riselyu manuho tersebu